# Gemini Video News Workflow

Build a short news video with Gemini agents and Veo: search for news, refine it, write a script, create a video prompt, then generate and download an MP4.

## 1. Install dependencies

In [ ]:
%pip install -qU openai-agents ddgs google-genai

## 2. Configure Gemini

Add `GEMINI_API_KEY` to Colab Secrets before running this cell. Video generation may take several minutes and requires Veo access for your Gemini API key.

In [ ]:
from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from google.colab import userdata
from openai import AsyncOpenAI

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
GEMINI_MODEL_NAME = "gemini-2.5-flash"
VIDEO_MODEL_NAME = "veo-3.1-generate-preview"

if not GEMINI_API_KEY:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets before running this notebook.")

set_tracing_disabled(disabled=True)
gemini_model = OpenAIChatCompletionsModel(
    model=GEMINI_MODEL_NAME,
    openai_client=AsyncOpenAI(
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key=GEMINI_API_KEY,
    ),
)

## 3. Find recent news

In [ ]:
from ddgs import DDGS


def search_news(query: str) -> str:
    """Find recent news and return titles, summaries, dates, and URLs."""
    results = DDGS(timeout=15).news(query, timelimit="d", max_results=10)
    if not results:
        raise RuntimeError("No recent news was found. Try a different topic.")

    for index, item in enumerate(results, start=1):
        print(f"{index}. {item.get('title', 'Untitled')}")
        print(item.get('url', 'No URL available.'))

    return "\n\n".join(
        f"Title: {item.get('title', 'Untitled')}\nDate: {item.get('date', 'Unknown')}\n"
        f"Summary: {item.get('body', 'No summary available.')}\nURL: {item.get('url', '')}"
        for item in results
    )


topic = "artificial intelligence business"
news_items = search_news(topic)

## 4. Refine the news and write a script

In [ ]:
from agents import Agent, Runner

editor_agent = Agent(
    name="News Editor",
    instructions="Select three relevant, credible, non-duplicative items. Exclude clickbait, ads, speculation, and weak evidence. Preserve the key facts, dates, and URLs.",
    model=gemini_model,
)
writer_agent = Agent(
    name="News Script Writer",
    instructions="Write a neutral 45- to 60-second spoken news script using only the supplied items. Do not add unsupported facts. End by naming the source publications without reading URLs aloud.",
    model=gemini_model,
)

editor_result = await Runner.run(editor_agent, f"Topic: {topic}\n\nNews items:\n{news_items}")
edited_news = editor_result.final_output
writer_result = await Runner.run(writer_agent, edited_news)
news_script = writer_result.final_output
print(news_script)

## 5. Video Director Agent

This new agent turns the news script into a concise Veo prompt. It plans visuals only; Veo performs the actual video generation.

In [ ]:
video_director_agent = Agent(
    name="Video Director",
    instructions=(
        "Create one Veo-ready prompt for an 8-second, landscape, editorial news B-roll video. "
        "Use only visual themes supported by the supplied script. Do not invent claims, include logos, "
        "show text overlays, name real people, or add spoken narration. Describe a simple sequence of 2–3 shots. "
        "Return only the video prompt."
    ),
    model=gemini_model,
)

director_result = await Runner.run(video_director_agent, news_script)
video_prompt = director_result.final_output
print(video_prompt)

## 6. Generate the video with Veo

Veo runs as a long-running job. This cell waits for completion, saves `news_brief.mp4`, and displays the result.

In [ ]:
import time

from google import genai
from IPython.display import Video, display

veo_client = genai.Client(api_key=GEMINI_API_KEY)
operation = veo_client.models.generate_videos(
    model=VIDEO_MODEL_NAME,
    prompt=video_prompt,
)

while not operation.done:
    print("Waiting for video generation to complete...")
    time.sleep(10)
    operation = veo_client.operations.get(operation)

if not operation.response or not operation.response.generated_videos:
    raise RuntimeError("Video generation did not return a video. Check Veo access and try again.")

video_path = "news_brief.mp4"
generated_video = operation.response.generated_videos[0]
veo_client.files.download(file=generated_video.video, destination=video_path)
display(Video(video_path, embed=True))

## 7. Download the MP4

In [ ]:
from google.colab import files

files.download(video_path)